In [1]:
# CELL 1: IMPORTS AND CONFIGURATION
import os
import random
import numpy as np
import pandas as pd
from PIL import Image
from tqdm.auto import tqdm
from datetime import datetime

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

from torchvision import transforms
from torchvision.models import efficientnet_b0, EfficientNet_B0_Weights

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, roc_auc_score

from torch.utils.tensorboard import SummaryWriter

# ================= CONFIG =================
SEED = 42

BATCH_SIZE = 256
EPOCHS = 10
LR = 3e-4
IMG_SIZE = 224

NUM_WORKERS = 4
PIN_MEMORY = True
PERSISTENT_WORKERS = True
PREFETCH_FACTOR = 4

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

torch.manual_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)

torch.backends.cudnn.benchmark = True
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

print("CUDA available:", torch.cuda.is_available())
print("GPU count:", torch.cuda.device_count())

2026-05-19 11:53:16.716349: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1779191596.740482    1053 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1779191596.748103    1053 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1779191596.768990    1053 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1779191596.769008    1053 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1779191596.769011    1053 computation_placer.cc:177] computation placer alr

CUDA available: True
GPU count: 2


In [2]:
# CELL 2: DATA COLLECTION
NO_PATHS = [
    "/kaggle/input/datasets/navoneel/brain-mri-images-for-brain-tumor-detection/no",
    "/kaggle/input/datasets/ahmedhamada0/brain-tumor-detection/no",
]

YES_PATHS = [
    "/kaggle/input/datasets/navoneel/brain-mri-images-for-brain-tumor-detection/yes",
    "/kaggle/input/datasets/ahmedhamada0/brain-tumor-detection/yes",
]

VALID_EXTENSIONS = (".jpg", ".jpeg", ".png", ".tif")


def collect(paths, label):
    items = []
    for path in paths:
        for root, _, files in os.walk(path):
            for f in files:
                if f.lower().endswith(VALID_EXTENSIONS):
                    full_path = os.path.join(root, f)
                    items.append((full_path, label))
    return items


data = collect(NO_PATHS, 0) + collect(YES_PATHS, 1)
df = pd.DataFrame(data, columns=["path", "label"])

print("Total Images:", len(df))
print("Class Distribution:")
print(df["label"].value_counts())

Total Images: 3253
Class Distribution:
label
1    1655
0    1598
Name: count, dtype: int64


In [ ]:
# VISUALIZATION: Sample Images from Dataset
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

fig, axes = plt.subplots(2, 8, figsize=(20, 5))
fig.suptitle("Sample Brain MRI Images (Top: No Tumor | Bottom: Tumor)", fontsize=14)

no_samples = df[df["label"] == 0].sample(8, random_state=SEED)
yes_samples = df[df["label"] == 1].sample(8, random_state=SEED)

for i, (_, row) in enumerate(no_samples.iterrows()):
    img = Image.open(row["path"]).convert("RGB").resize((224, 224))
    axes[0, i].imshow(img)
    axes[0, i].axis("off")
    axes[0, i].set_title("No Tumor", fontsize=8)

for i, (_, row) in enumerate(yes_samples.iterrows()):
    img = Image.open(row["path"]).convert("RGB").resize((224, 224))
    axes[1, i].imshow(img)
    axes[1, i].axis("off")
    axes[1, i].set_title("Tumor", fontsize=8)

plt.tight_layout()
plt.show()

In [4]:
# CELL 3: TRAIN/VAL/TEST SPLIT
train_df, temp_df = train_test_split(
    df, test_size=0.3, stratify=df["label"], random_state=SEED
)

val_df, test_df = train_test_split(
    temp_df, test_size=0.5, stratify=temp_df["label"], random_state=SEED
)

print("Train:", len(train_df))
print("Val:", len(val_df))
print("Test:", len(test_df))

Train: 2277
Val: 488
Test: 488


In [ ]:
# VISUALIZATION: Class Distribution Across Splits
import matplotlib.pyplot as plt
import numpy as np

splits = ["Train", "Val", "Test"]
dfs = [train_df, val_df, test_df]
no_counts = [d[d["label"] == 0].shape[0] for d in dfs]
yes_counts = [d[d["label"] == 1].shape[0] for d in dfs]

x = np.arange(len(splits))
width = 0.35

fig, ax = plt.subplots(figsize=(8, 5))
ax.bar(x - width/2, no_counts, width, label="No Tumor", color="steelblue")
ax.bar(x + width/2, yes_counts, width, label="Tumor", color="tomato")
ax.set_xticks(x)
ax.set_xticklabels(splits)
ax.set_ylabel("Count")
ax.set_title("Class Distribution per Split")
ax.legend()
for i, (n, y) in enumerate(zip(no_counts, yes_counts)):
    ax.text(i - width/2, n + 1, str(n), ha="center", va="bottom", fontsize=9)
    ax.text(i + width/2, y + 1, str(y), ha="center", va="bottom", fontsize=9)
plt.tight_layout()
plt.show()

In [6]:
# CELL 4: DATASET CLASS
class BrainDataset(Dataset):
    def __init__(self, df, transform=None):
        self.df = df.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        img_path = self.df.loc[idx, "path"]
        label = self.df.loc[idx, "label"]

        try:
            img = Image.open(img_path).convert("RGB")
        except Exception as e:
            raise RuntimeError(f"Failed loading image: {img_path}") from e

        if self.transform:
            img = self.transform(img)

        return img, torch.tensor(label, dtype=torch.float32)

In [7]:
# CELL 5: TRANSFORMS AND DATALOADERS
train_tfms = transforms.Compose(
    [
        transforms.Resize((IMG_SIZE, IMG_SIZE)),
        transforms.RandomHorizontalFlip(),
        transforms.RandomRotation(10),
        transforms.ColorJitter(0.2, 0.2),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
    ]
)

val_tfms = transforms.Compose(
    [
        transforms.Resize((IMG_SIZE, IMG_SIZE)),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
    ]
)

train_loader = DataLoader(
    BrainDataset(train_df, train_tfms),
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
    persistent_workers=PERSISTENT_WORKERS,
    prefetch_factor=PREFETCH_FACTOR,
    drop_last=True,
)

val_loader = DataLoader(
    BrainDataset(val_df, val_tfms),
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
    persistent_workers=PERSISTENT_WORKERS,
    prefetch_factor=PREFETCH_FACTOR,
    drop_last=False,
)

test_loader = DataLoader(
    BrainDataset(test_df, val_tfms),
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
    persistent_workers=PERSISTENT_WORKERS,
    prefetch_factor=PREFETCH_FACTOR,
    drop_last=False,
)

In [ ]:
# VISUALIZATION: Augmentation Comparison (Original vs Augmented)
import matplotlib.pyplot as plt
import torchvision.transforms.functional as TF

sample_paths = train_df[train_df["label"] == 1].sample(4, random_state=SEED)["path"].tolist()

basic_tfm = transforms.Compose([transforms.Resize((IMG_SIZE, IMG_SIZE))])
aug_tfm = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(p=1.0),
    transforms.RandomRotation(10),
    transforms.ColorJitter(0.2, 0.2),
])

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
fig.suptitle("Original (Top) vs Augmented (Bottom)", fontsize=14)

for i, path in enumerate(sample_paths):
    img = Image.open(path).convert("RGB")
    axes[0, i].imshow(basic_tfm(img))
    axes[0, i].axis("off")
    axes[0, i].set_title("Original", fontsize=9)
    axes[1, i].imshow(aug_tfm(img))
    axes[1, i].axis("off")
    axes[1, i].set_title("Augmented", fontsize=9)

plt.tight_layout()
plt.show()

In [9]:
# CELL 6: MODEL INITIALIZATION
model = efficientnet_b0(weights=EfficientNet_B0_Weights.DEFAULT)
model.classifier[1] = nn.Linear(model.classifier[1].in_features, 1)

if torch.cuda.device_count() > 1:
    print("Using", torch.cuda.device_count(), "GPUs")
    model = nn.DataParallel(model)

model = model.to(DEVICE)

criterion = nn.BCEWithLogitsLoss()
optimizer = optim.AdamW(model.parameters(), lr=LR)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

scaler = torch.amp.GradScaler("cuda", enabled=torch.cuda.is_available())

Using 2 GPUs


In [10]:
# CELL 7: TRAIN/VALIDATION LOOP FUNCTION
def run_epoch(loader, training=True):
    if training:
        model.train()
    else:
        model.eval()

    total_loss = 0.0
    all_preds = []
    all_labels = []

    loop = tqdm(
        loader, total=len(loader), desc="Train" if training else "Val/Test", leave=False
    )

    for images, labels in loop:
        use_nb = not isinstance(model, torch.nn.DataParallel)
        images = images.to(DEVICE, non_blocking=use_nb)
        labels = labels.to(DEVICE, non_blocking=use_nb)

        if training:
            optimizer.zero_grad(set_to_none=True)

        with torch.amp.autocast(device_type=DEVICE.type, enabled=torch.cuda.is_available()):
            outputs = model(images).squeeze(1)
            loss = criterion(outputs, labels)

        if training:
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()

        total_loss += loss.item()
        probs = torch.sigmoid(outputs).detach()
        all_preds.append(probs.cpu())
        all_labels.append(labels.detach().cpu())
        loop.set_postfix(loss=f"{loss.item():.4f}")

    all_preds = torch.cat(all_preds).numpy()
    all_labels = torch.cat(all_labels).numpy()

    acc = accuracy_score(all_labels, all_preds > 0.5)
    auc = roc_auc_score(all_labels, all_preds)

    return total_loss / len(loader), acc, auc

In [11]:
# CELL 8: TRAINING LOOP WITH TENSORBOARD
log_dir = f"/kaggle/working/runs/{datetime.now().strftime('%Y%m%d-%H%M%S')}"
writer = SummaryWriter(log_dir)

best_auc = 0.0
history = {"train_loss": [], "val_loss": [], "train_auc": [], "val_auc": [], "train_acc": [], "val_acc": []}

for epoch in range(EPOCHS):
    print(f"\n===== Epoch {epoch+1}/{EPOCHS} =====")

    train_loss, train_acc, train_auc = run_epoch(train_loader, True)
    val_loss, val_acc, val_auc = run_epoch(val_loader, False)

    scheduler.step()

    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)
    history["train_auc"].append(train_auc)
    history["val_auc"].append(val_auc)
    history["train_acc"].append(train_acc)
    history["val_acc"].append(val_acc)

    print(
        f"Train | Loss: {train_loss:.4f} | Acc: {train_acc:.4f} | AUC: {train_auc:.4f}"
    )
    print(f"Val   | Loss: {val_loss:.4f} | Acc: {val_acc:.4f} | AUC: {val_auc:.4f}")

    writer.add_scalar("Loss/Train", train_loss, epoch)
    writer.add_scalar("Loss/Val", val_loss, epoch)
    writer.add_scalar("AUC/Train", train_auc, epoch)
    writer.add_scalar("AUC/Val", val_auc, epoch)

    if val_auc > best_auc:
        best_auc = val_auc
        state_dict = (
            model.module.state_dict()
            if torch.cuda.device_count() > 1
            else model.state_dict()
        )
        torch.save(state_dict, "/kaggle/working/best_efficientnet.pth")
        print("Best model saved.")

writer.close()


===== Epoch 1/10 =====


Train:   0%|          | 0/8 [00:00<?, ?it/s]

Val/Test:   0%|          | 0/2 [00:00<?, ?it/s]

Train | Loss: 0.4974 | Acc: 0.8115 | AUC: 0.9039
Val   | Loss: 0.3447 | Acc: 0.8586 | AUC: 0.9686
Best model saved.

===== Epoch 2/10 =====


Train:   0%|          | 0/8 [00:00<?, ?it/s]

...

Val/Test:   0%|          | 0/2 [00:00<?, ?it/s]

Train | Loss: 0.0067 | Acc: 0.9990 | AUC: 1.0000
Val   | Loss: 0.0195 | Acc: 0.9939 | AUC: 0.9998

===== Epoch 10/10 =====


Train:   0%|          | 0/8 [00:00<?, ?it/s]

Val/Test:   0%|          | 0/2 [00:00<?, ?it/s]

Train | Loss: 0.0055 | Acc: 0.9995 | AUC: 1.0000
Val   | Loss: 0.0211 | Acc: 0.9939 | AUC: 0.9998


In [ ]:
# VISUALIZATION: Training Curves
import matplotlib.pyplot as plt

epochs_range = range(1, EPOCHS + 1)
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle("Training History", fontsize=14)

axes[0].plot(epochs_range, history["train_loss"], label="Train", marker="o")
axes[0].plot(epochs_range, history["val_loss"], label="Val", marker="o")
axes[0].set_title("Loss")
axes[0].set_xlabel("Epoch")
axes[0].legend()

axes[1].plot(epochs_range, history["train_auc"], label="Train", marker="o")
axes[1].plot(epochs_range, history["val_auc"], label="Val", marker="o")
axes[1].set_title("AUC")
axes[1].set_xlabel("Epoch")
axes[1].legend()

axes[2].plot(epochs_range, history["train_acc"], label="Train", marker="o")
axes[2].plot(epochs_range, history["val_acc"], label="Val", marker="o")
axes[2].set_title("Accuracy")
axes[2].set_xlabel("Epoch")
axes[2].legend()

plt.tight_layout()
plt.show()

In [13]:
# CELL 9: TESTING
state_dict = torch.load("/kaggle/working/best_efficientnet.pth")

if torch.cuda.device_count() > 1:
    model.module.load_state_dict(state_dict)
else:
    model.load_state_dict(state_dict)

model.eval()
test_loss, test_acc, test_auc = run_epoch(test_loader, training=False)

print("\n===== FINAL UNSEEN TEST RESULTS =====")
print(f"Test Loss: {test_loss:.4f}")
print(f"Test Accuracy: {test_acc:.4f}")
print(f"Test AUC: {test_auc:.4f}")

Val/Test:   0%|          | 0/2 [00:00<?, ?it/s]


===== FINAL UNSEEN TEST RESULTS =====
Test Loss: 0.0276
Test Accuracy: 0.9898
Test AUC: 0.9998


In [14]:
# VISUALIZATION: Test Results — Confusion Matrix, ROC Curve, Prediction Samples
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, roc_curve, ConfusionMatrixDisplay

# Collect all test predictions
model.eval()
all_preds_test = []
all_labels_test = []
all_paths_test = []

test_dataset = BrainDataset(test_df.reset_index(drop=True), val_tfms)

with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(DEVICE)
        
        outputs = model(images).squeeze(1)
        probs = torch.sigmoid(outputs).cpu().numpy()
        all_preds_test.extend(probs)
        all_labels_test.extend(labels.numpy())

all_preds_test = np.array(all_preds_test)
all_labels_test = np.array(all_labels_test)
binary_preds = (all_preds_test > 0.5).astype(int)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Confusion matrix
cm = confusion_matrix(all_labels_test, binary_preds)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["No Tumor", "Tumor"])
disp.plot(ax=axes[0], colorbar=False, cmap="Blues")
axes[0].set_title("Confusion Matrix (Test Set)")

# ROC curve
fpr, tpr, _ = roc_curve(all_labels_test, all_preds_test)
from sklearn.metrics import auc as sklearn_auc
roc_auc_val = sklearn_auc(fpr, tpr)
axes[1].plot(fpr, tpr, color="darkorange", lw=2, label=f"AUC = {roc_auc_val:.4f}")
axes[1].plot([0, 1], [0, 1], color="navy", lw=1, linestyle="--")
axes[1].set_xlim([0.0, 1.0])
axes[1].set_ylim([0.0, 1.05])
axes[1].set_xlabel("False Positive Rate")
axes[1].set_ylabel("True Positive Rate")
axes[1].set_title("ROC Curve (Test Set)")
axes[1].legend(loc="lower right")

plt.tight_layout()
plt.show()

# Prediction samples — 4 correct, 4 incorrect
correct_idx = np.where(binary_preds == all_labels_test)[0]
wrong_idx = np.where(binary_preds != all_labels_test)[0]

n_correct = min(4, len(correct_idx))
n_wrong = min(4, len(wrong_idx))

fig2, axes2 = plt.subplots(2, max(n_correct, n_wrong, 1), figsize=(16, 8))
fig2.suptitle("Correct Predictions (Top) | Wrong Predictions (Bottom)", fontsize=13)

for i in range(max(n_correct, n_wrong)):
    for row, idx_arr, label_prefix in [(0, correct_idx, "✓"), (1, wrong_idx, "✗")]:
        ax = axes2[row, i] if max(n_correct, n_wrong) > 1 else axes2[row]
        if i < len(idx_arr):
            idx = idx_arr[i]
            path = test_df.reset_index(drop=True).loc[idx, "path"]
            true_lbl = int(all_labels_test[idx])
            pred_lbl = int(binary_preds[idx])
            prob = all_preds_test[idx]
            img = Image.open(path).convert("RGB").resize((224, 224))
            ax.imshow(img)
            ax.set_title(
                f"{label_prefix} True: {'Tumor' if true_lbl else 'No'} | Pred: {prob:.2f}",
                fontsize=8, color="green" if row == 0 else "red"
            )
            ax.axis("off")
        else:
            ax.axis("off")

plt.tight_layout()
plt.show()

AcceleratorError: Caught AcceleratorError in replica 0 on device 0.
Original Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/nn/parallel/parallel_apply.py", line 103, in _worker
    output = module(*input, **kwargs)
             ^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/torch/nn/modules/module.py", line 1776, in _wrapped_call_impl
    return self._call_impl(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/torch/nn/modules/module.py", line 1787, in _call_impl
    return forward_call(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/torchvision/models/efficientnet.py", line 344, in forward
    return self._forward_impl(x)
           ^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/torchvision/models/efficientnet.py", line 334, in _forward_impl
    x = self.features(x)
        ^^^^^^

In [ ]:
# CELL 10: SAVE FINAL MODEL
final_state = (
    model.module.state_dict() if torch.cuda.device_count() > 1 else model.state_dict()
)
torch.save(final_state, "/kaggle/working/final_efficientnet_brain_tumor.pth")

print("Final model saved.")